# Examples of Pyspark ML

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName("MLExample").getOrCreate()

##### Read the dataset

In [4]:
training = spark.read.csv("../Datasets/test3.csv", header=True, inferSchema=True)

In [5]:
training.show()

+-----+---+----------+------+
| Name|Age|Experience|Salary|
+-----+---+----------+------+
|Krish| 31|        10|300000|
|Alexa| 30|         8|250000|
|Sunny| 29|         4|200000|
| Paul| 24|         3| 50000|
| Siri| 21|         1| 20000|
|James| 23|         2| 30000|
+-----+---+----------+------+



In [6]:
training.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [7]:
training.columns

['Name', 'Age', 'Experience', 'Salary']

In [9]:
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols=['age','Experience'],outputCol="Independent Features")

In [10]:
output = assembler.transform(training)

In [11]:
output.show()

+-----+---+----------+------+--------------------+
| Name|Age|Experience|Salary|Independent Features|
+-----+---+----------+------+--------------------+
|Krish| 31|        10|300000|         [31.0,10.0]|
|Alexa| 30|         8|250000|          [30.0,8.0]|
|Sunny| 29|         4|200000|          [29.0,4.0]|
| Paul| 24|         3| 50000|          [24.0,3.0]|
| Siri| 21|         1| 20000|          [21.0,1.0]|
|James| 23|         2| 30000|          [23.0,2.0]|
+-----+---+----------+------+--------------------+



In [12]:
output.columns

['Name', 'Age', 'Experience', 'Salary', 'Independent Features']

In [13]:
finalized_data = output.select("Independent Features","Salary")

In [14]:
finalized_data.show()

+--------------------+------+
|Independent Features|Salary|
+--------------------+------+
|         [31.0,10.0]|300000|
|          [30.0,8.0]|250000|
|          [29.0,4.0]|200000|
|          [24.0,3.0]| 50000|
|          [21.0,1.0]| 20000|
|          [23.0,2.0]| 30000|
+--------------------+------+



In [22]:
from pyspark.ml.regression import LinearRegression
# train test split
train_data, test_data = finalized_data.randomSplit([0.75,0.25])
regressor = LinearRegression(featuresCol="Independent Features", labelCol="Salary")
regressor = regressor.fit(train_data)

In [23]:
# Coefficients
regressor.coefficients

DenseVector([19912.2807, 14561.4035])

In [24]:
# Intercepts
regressor.intercept

-463859.64912267606

In [25]:
# Prediction
pred_results = regressor.evaluate(dataset=test_data)

In [26]:
pred_results.predictions.show()

+--------------------+------+-------------------+
|Independent Features|Salary|         prediction|
+--------------------+------+-------------------+
|          [21.0,1.0]| 20000|-31140.350877186807|
|          [29.0,4.0]|200000|  171842.1052631337|
+--------------------+------+-------------------+



In [27]:
pred_results.meanAbsoluteError, pred_results.meanSquaredError, pred_results.rootMeanSquaredError

(39649.12280702655, 1704101261.9271119, 41280.76140197891)